# Thermostat with Hysteresis - Hybrid Automaton Example

This notebook demonstrates a hybrid automaton modeling a traffic light controller.

**States**: HEATING → IDL → COOLING

**Transitions** occur based on time thresholds:
- RED stays for 5 seconds
- GREEN stays for 8 seconds
- YELLOW stays for 3 seconds

## 1. Import Required Modules

In [17]:
from hybrid_automaton import Automaton, State, Transition
import time
import numpy as np

## 2. Define Guard Functions

Guards check if enough time has elapsed to trigger a transition.

In [18]:
def too_cold(x, aux_x, u, ctx, dt):
    return x['temp'] < 18.0

def too_hot(x, aux_x, u, ctx, dt):
    return x['temp'] > 22.0

def temp_comfortable(x, aux_x, u, ctx, dt):
    return 18.5 <= x['temp'] <= 21.5

## 3. Define Dynamics

The dynamic function for each state.

In [27]:
def heating_dynamics(x, aux_x, u, ctx, dt):
    """Temperature increases when heating"""
    return 2.0  # Heating rate: 2°C/s

def cooling_dynamics(x, aux_x, u, ctx, dt):
    """Temperature decreases when cooling"""
    return -1.5  # Cooling rate: -1.5°C/s

def idle_dynamics(x, aux_x, u, ctx, dt):
    """Temperature drifts toward ambient (20°C)"""
    ambient = 20.0
    drift_rate = 0.5
    return drift_rate * (ambient - x['temp'])

## 4. Define Callback Functions

callback functions for on entry of each of the states

In [20]:
def on_enter_heating():
    print("🔥 HEATING - Turning heater ON")

def on_enter_cooling():
    print("❄️  COOLING - Turning AC ON")

def on_enter_idle():
    print("😌 IDLE - Systems off, maintaining temperature")

## 5. Define States

In [21]:
heating = State(name="HEATING", flow=heating_dynamics, on_enter=on_enter_heating)
cooling = State(name="COOLING", flow=cooling_dynamics, on_enter=on_enter_cooling)
idle = State(name="IDLE", initial=True, flow=idle_dynamics, on_enter=on_enter_idle)

## 6. Define Transitions

In [22]:
idle.add_transition(Transition("idle_to_heating", heating, guards=[too_cold], priority=1))
idle.add_transition(Transition("idle_to_cooling", cooling, guards=[too_hot], priority=1))

heating.add_transition(Transition("heating_to_idle", idle, guards=[temp_comfortable], priority=1))
cooling.add_transition(Transition("cooling_to_idle", idle, guards=[temp_comfortable], priority=1))

## 7. Create Automaton

In [23]:
thermostat = Automaton(
    name="Thermostat",
    states=[heating, cooling, idle],
    dt=0.1,
    real_time_mode=False
)

# Initial state: temperature at 15°C (cold)
x0 = {'temp': 15.0}
thermostat.activate(x0=x0)

## 8. Run the Simulation 

In [29]:
for i in range(200):
    result = thermostat.step()
    if i % 20 == 0:
        print(f"t={i*0.1:.1f}s | State: {thermostat.q.name:8s} | Temp: {thermostat.x['temp']:.2f}°C")
    if result and result.transition_taken:
        print(f"  → Transition: {result.transition_taken.name}")

print(f"\nFinal temperature: {thermostat.x['temp']:.2f}°C")
print()

TypeError: unsupported operand type(s) for *: 'dict' and 'float'